# Domain-Adaptable Speech-to-Text — End-to-End Training 

This notebook runs the full pipeline:
1. Clone repo / mount Drive for checkpoint persistence
2. Install dependencies
3. Generate synthetic domain prompts
4. Synthesize audio (TTS)
5. Augment with noise
6. Prepare HF dataset (mel-spectrogram features + tokenized labels)
7. LoRA fine-tune Whisper-small
8. Evaluate WER: baseline vs fine-tuned, clean vs noisy



In [ ]:
# 0. Mount Google Drive so checkpoints/data survive Colab disconnects
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/foundry-speech-to-text'
import os
os.makedirs(PROJECT_DIR, exist_ok=True)
%cd {PROJECT_DIR}

In [ ]:
# 1. Clone the repo (replace with your actual GitHub URL once pushed)
!git clone https://github.com/Yamini1727/foundry-speech-to-text.git repo
%cd repo

In [ ]:
# 2. Install dependencies
!pip install -q -r requirements.txt

In [ ]:
# 3. Generate prompts
!python data/generate_prompts.py --output data/prompts.json --n_general 150 --n_domain 200 --n_generic_obs 100

In [ ]:
# 4. Synthesize audio via TTS (this is the slowest step - grab a coffee)
!python data/synthesize_audio.py --prompts data/prompts.json --output_dir data/audio_raw

In [ ]:
# 5. Augment with machinery/industrial noise (ESC-50, free CC-licensed dataset)
!python data/augment_noise.py --manifest data/audio_raw/manifest.json --output_dir data/audio_augmented

In [ ]:
# 6. Prepare HF dataset (feature extraction + tokenization)
!python training/prepare_dataset.py --manifest data/audio_augmented/manifest_full.json \
    --output_dir hf_dataset --model_name openai/whisper-small

In [ ]:
# 7. LoRA fine-tune (the actual training step)
!python training/finetune_lora.py --dataset_dir hf_dataset --output_dir whisper-lora-foundry \
    --model_name openai/whisper-small --epochs 4 --batch_size 8

In [ ]:
# 8. Evaluate: baseline vs fine-tuned, clean vs noisy WER
!python training/evaluate.py --dataset_dir hf_dataset --base_model openai/whisper-small \
    --lora_adapter whisper-lora-foundry --output results/wer_comparison.json

In [ ]:
# 9. (Optional) Quick sanity-check transcription on a single held-out sample
import json
with open('results/wer_comparison.json') as f:
    results = json.load(f)
for sample in results['sample_predictions'][:5]:
    print('REF:', sample['reference'])
    print('BASELINE:', sample['baseline_pred'])
    print('FINE-TUNED:', sample['finetuned_pred'])
    print('---')